# Preparations

## Imports

In [4]:
import polars as pl

playlists = pl.scan_parquet('../processed_data/data_playlist_metadata.parquet')
playlist_tracks = pl.scan_parquet('../processed_data/data_playlist_songs.parquet')
tracks = pl.scan_parquet('../processed_data/data_song_metadata.parquet')

# Analysis

## Patterns

### Load keywords for tagging from files

The keywords (and any aliases that should be mapped onto the same keyword) are stored in YAML files for better mainteneability.

In [35]:
from typing import TypedDict

import yaml

type _KeywordEntry = str | dict[str, str | None | list[_KeywordEntry]]

def _append_to[Key, Value](mapping: dict[Key, set[Value]], key: Key, values: list[Value]):
    if key in mapping:
        mapping[key].update(values)
    else:
        mapping[key] = set(values)

def _format_tag(category: str, name: str) -> str:
    return f'{category}:{name}'

def _traverse_entry(entry: _KeywordEntry, category: str, tags: set[str], result: dict[str, set[str]], use_as_tag: bool = False):
    """Visit the given `entry` and its children, and add the resulting word-to-alias mappings to `result`."""
    if isinstance(entry, str):
        # keywords:
        #   genre:
        #     - pop # <--
        _append_to(result, entry, tags if not use_as_tag else
                   [*tags, _format_tag(category, entry)])
    elif isinstance(entry, dict):
        # keywords:
        #   genre:
        #     - acoustic: # <--
        #       ...
        for tag_spec in entry:
            # Tag names can have certain modifiers
            tag, is_lower_weight = tag_spec, False

            # Adding a question mark "?" to the end of a tag indicates
            # that its child entries might be only imprecise matches
            # TODO: Actually do something with that information
            if tag.endswith("?"):
                tag, is_lower_weight = tag[:-1], True

            children = entry[tag_spec]
            child_tags = [*tags, _format_tag(category, tag)]
            if children is None:
                # keywords:
                #   genre:
                #     - acoustic:
                _append_to(result, tag, tags)
            elif isinstance(children, str):
                # keywords:
                #   genre:
                #     - poprock: pop-rock
                _append_to(result, children, tags)
            elif isinstance(children, list):
                # keywords:
                #   genre:
                #     - acoustic: # <--
                #         - acoustic
                #         - acoustics
                for child in children:
                    _traverse_entry(child, category, child_tags, result)
            else:
                raise TypeError("Neither a str nor a list nor None")
    else:
        raise TypeError(f"Neither a str nor a dict")

class _KeywordsFile(TypedDict):
    keywords: dict[str, list[_KeywordEntry]]

def load_keyword_aliases(category_as_tag: bool = False):
    with open("../utils/keyword_data.yaml") as stream:
        raw_data: _KeywordsFile = yaml.safe_load(stream)

    _aliases: dict[str, set[str]] = {}
    for category in raw_data['keywords']:
        for entry in raw_data['keywords'][category]:
            _traverse_entry(entry, category, [category] if category_as_tag else [], _aliases, True)

    return {k: list(_aliases[k]) for k in _aliases}


aliases = load_keyword_aliases()

aliases

{'comp': ['context:competition'],
 'competition': ['context:competition'],
 'comps': ['context:competition'],
 'contest': ['context:competition'],
 'final': ['context:competition'],
 'finals': ['context:competition'],
 'heat': ['context:competition'],
 'invitational': ['context:competition'],
 'j&j': ['context:competition'],
 'jill': ['context:competition'],
 'jnj': ['context:competition'],
 'prelim': ['context:competition'],
 'prelims': ['context:competition'],
 'semis': ['context:competition'],
 'spotlight': ['context:competition'],
 'spotlights': ['context:competition'],
 'strictly': ['context:competition'],
 'bootcamp': ['context:bootcamp', 'context:class'],
 'class': ['context:class'],
 'classes': ['context:class'],
 'cours': ['context:class'],
 'course': ['context:class'],
 'group': ['context:class'],
 'group class': ['context:class'],
 'kurs': ['context:class'],
 'l1': ['context:class'],
 'l2': ['context:class'],
 'l3': ['context:class'],
 'learn': ['context:class'],
 'lesson': 

### Additional patterns

In [13]:
patterns: dict[str, list[str]] = {}

#### Context & Level

#### Music

In [14]:
patterns["season"] = pattern_seasons = [
    # Birthdays
    "b-day",
    "bday",
    "birthday",

    # Seasons
    "autumn",
    "fall",
    "spring",  # also matches certain events
    "summer",  # also matches certain events
    "winter",  # also matches certain events

    # More
    "easter",
    "holiday",
    "season",
    "spooky",
    "thanksgiving",
    "valentine's",
    "valentine",
    "valentines",

    # Even more
    "beach",
    "pool",
    "silvester",
    "anniversary",
]

In [15]:
patterns["topic"] = pattern_topics = [
    # "connection", # needs further filtering - "Connection Nürnberg"
]

In [16]:
patterns["dance"] = pattern_dances = [
    # West Coast Swing
    "wc", "wcs", "(wcs)", "wcs-", "wsc", "wcs:", "wcs.",
    "west", "coast", "swing",
    "westcoast", "westcoastswing",
    "westie:", "westie", "westies",

    # Ballroom & Latin
    "ccc",  # "cha cha", "cha cha cha",
    "cha",  # Review
    "df", "discofox",
    "lw",
    "nc2", "nc2s", "two-step",  # "nightclub twostep",
    "qs",
    "quik step",
    "quikstep",
    "quick step",
    "quickstep",
    "rb", "ru", "rumba",
    "sf",
    "tango", "argentino", "milonga",
    "ww",  # imprecise?

    # More ballroom
    "ball",
    "chartbreaker",
    "hochzeit",
    "tanz",
    "tanzmusik",
    "tanzorchester",
    "tanz orchester",
    "klaus hallen"
    "tanzparty",
    "tanzrunde",

    # Unrelated solo dancing
    "kita", "kids",
    "hh",
    "fitness",
    "hardstyle",
    "jumpstyle",
    "pole",  # "pole dance", "pole dancing",
    "line",  # "line dance"

    # Other dance styles
    "crossover",
    "kiz", "kizomba",  # "urban kiz",
    "soltinho",  # Google says: "Brasilian couple dance, that developed from East Coast Swing and is danced to Rock, Disco and Swing."
    "swouk",
    "zouk",
]

In [17]:
patterns["language"] = pattern_languages = [
    # Note: We likely want to filter out stuff like "French Open", "German Open"
    #       when we are trying to find e.g. "German [Songs]"
    "chinese",
    "english",
    "french", "française", "français",
    "german", "deutsch",
    "korean",
    "spanish",
]

#### Exploration

In [18]:
patterns["weird"] = pattern_weird = [
    "odd",
    "weird",
    "strange",
]

#### Skip words

In [19]:
patterns["too_broad"] = pattern_too_broad = [
    "dance", "dans", "danse", "dancing",
    "drinking",
    "love",
    "remix",
    "best",  # "best of",
    "live",
    "open",
    "freestyle",
    "critique",
    "rounds",
]

In [20]:
patterns["too_generic"] = pattern_too_generic = [
    "(deluxe", "(deluxe)",
    "album", "release", "various", "artists",
    "alt", "backup",
    "baby",
    "battle",  # "dj battle",
    "beats",
    "bpm", "bpm:", "bpm)", "bpm]", "rpm",
    "cast",
    "collab",
    "dj", "djs", "[dj",
    "edition", "edition)",
    "friends",
    "game", "games",
    "general",
    "island",
    "jam", "jams", "jamz",  # ?
    "mini",
    "movie", "video",
    "music", "musik",
    "original", "(original",
    "part", "pt", "vol.", "volume",
    "room", "hall",
    "sélection", "shortlist", "options", "collection", "bag", "jukebox",
    "set", "setlist", "played", "playlist", "playlista", "list", "lista", "mix",
    "song", "songs", "titres", "tunes", "tracks", "låtar",
    "sound", "sounds",
    "spotify",
    "thing", "these",
    "time",
    "version", "version)",

    # Maybe indicates a party?
    "weekly",
    "century",
    "year", "years",
    "monthly",
    "week",
    "daily", "day", "days",
    "today", "tonight",
    "afternoon",
    "first", "second", "half",
    "hour",
    "end",
]

#### Months & Weekdays

In [21]:
patterns["weekday"] = pattern_weekdays = [
    "mon", "mon,",
    "monday", "monday,",
    "tue", "tue,", "tues",
    "tuesday", "tuesday,",
    "tuesdays",
    "wed", "wed,",
    "wednesday", "wednesday,",
    "thu", "thu,", "thurs",
    "thursday", "thursday,",
    "fri", "fri,",
    "friday", "friday,",
    "sat", "sat,",
    "saturday", "saturday,",
    "sun", "sun,",
    "sunday", "sunday,",

    # French
    "dimanche",
    "jeudi",
    "lundi",
    "mardi",
    "samedi",
    "vendredi",
]

In [22]:
patterns["month"] = pattern_months = [
    "jan", "january", "januar", "janvier",
    "feb", "february", "februar", "février",
    "mar", "march", "märz", "mars",
    "apr", "april", "avril",
    "may", "mai",
    "jun", "june", "juni", "juin",
    "jul", "july", "juli", "juillet",
    "aug", "august", "août",
    "sep", "sept", "september", "septembre",
    "oct", "october", "okt", "oktober",
    "nov", "november", "novembre",
    "dec", "december", "dez", "dezember",
]

#### DJ/Artist/Event Names

In [23]:
patterns["dj"] = pattern_djs = [
    "alex",
    "andrzejki",
    "artur",
    "attucks",
    "beti",
    "breno",
    "david",
    "michaela",
    "dyos",
    "fuzz",
    "kia",  # "dj kia",
    "lew",
    "lils",
    "lojyk's",
    "marco",
    "margies",
    "matt",
    "meech",
    "poronin",
    "psdj",
    "rick",
    "ruby",
    "sam's",
    "sara",  # "dj sara",
    "shay",
    "snail]",
    "steve",
    "tein",
    "tim",
    "wah",
    "yvonne",
]

In [24]:
patterns["artist"] = pattern_artists = [
    # Artist names
    "bryan", "jordan",
    "coldplay",
    "ed", "sheeran",
    "greatest", "showman",
    "imagine", "dragons",
    "john", "mayer",
    "michael", "jackson",
    "mumford", "sons",
    "sam", "smith",
    "sara", "bareilles",
    "taylor", "swift",

    # Unsorted
    "amigo's",
    "billy",
    "brad",
    "chris",
    "jack",
    "james",
    "jason",
    "justin",
    "linkin",
    "prince",
    "shawn",
    "train",
    "williams",
]

In [25]:
patterns["language"] = pattern_languages = [
    # Note: We likely want to filter out stuff like "French Open", "German Open"
    #       when we are trying to find e.g. "German [Songs]"
    "chinese",
    "english",
    "french", "française", "français",
    "german", "deutsch",
    "korean",
    "spanish",
]

In [26]:
patterns["maybe_events"] = pattern_maybe_events_or_organizers = [
    "mcs",
    "angels",
    "antonio",
    "aw",
    "awa",
    "baltic",
    "barka",  # ?
    "bash",
    "bcsdc",
    "berlin",
    "bh",
    "boston",
    "botb",
    "brunch",
    "bsl",
    "budafest",
    "buli",
    "buma",
    "całonocna",
    "capital",  # Capital Swing
    "central",
    "chatt",
    "chicago",
    "city",  # imprecise # Westie Pink City, but also Owl City
    "classic",  # imprecise # Paris Swing Classic, Tx Classic Swing
    "club",  # imprecise # Tyrol Club of Solvay
    "code",  # imprecise?
    "collective",
    "college",
    "crush",
    "danceboston",
    "dc",
    "dcsx",
    "diego",  # Swing Diego, San Diego
    "dual",  # imprecise
    "dv",
    "elks",
    "empower",  # XPRESS EMPOWER
    "esdc",
    "farnham",
    "fest",
    "festival",
    "ff",
    "five",  # "friday five",
    "fling",
    "float",
    "fnpl",
    "fools",  # Dancing Fools
    "haifa",
    "hcs",
    "idance",  # iDance
    "infused",
    "itsallswing",
    "jj",
    "k&s",
    "keller",
    "lab",  # "Dance Lab",
    "ladc",
    "liberty",
    "madjam",  # "MADjam",
    "madness",
    "mamalist",
    "mj",
    "mwcsc",
    "mwf",
    "nordic",  # Nordic Open
    "nye",
    "nzo",
    "osaka",
    "paris",
    "phoenix",
    "pink",
    "potsdam",
    "proswingdjs",
    "push",
    "rc",
    "red",
    "roc",  # Wild Westies ROC
    "rocket",
    "rose",
    "rtb",
    "rx",
    "ryan",
    "san",
    "sea",
    "seattle",
    "shakedown",
    "silbando",
    "sofia",
    "ss",
    "ssdc",
    "st",
    "stanford",
    "street",
    "strength",  # imprecise
    "studio",
    "summit",
    "sundance",
    "swingesota",
    "swingout",  # SwingOut
    "swingsation",
    "swingside",
    "swingtacular",
    "swingtzerland",
    "switchx",  # SwitchXperience
    "synergy",
    "syracuse",  # Syracuse Socials
    "tap",  # The After Party
    "tb",
    "tds",
    "tp",
    "tsp",
    "ucswing",
    "uk",
    "ulm",
    "uptown",
    "usa",
    "wasda",
    "wcs@home",
    "wcsa",
    "wcw",
    "westiebos",
    "westival",
    "westy",
    "white",
    "whs",
    "wicked",
    "wild",
    "wotp",  # Westie On The Promenade
    "xchange",
    "xpress",  # XPRESS CLASSIC, XPRESS CONNECT
    "żaczek",
    "zf", "konobueno",
    "zonawcs",
]

## Tokenization / Keyword mapping

Generic tokenization for single-word terms:

In [31]:
def tokenize(expr: pl.Expr) -> pl.Expr:
    return expr.str.to_lowercase().str.split(' ')


def tokenize_unique(expr: pl.Expr) -> pl.Expr:
    return tokenize(expr)\
        .list.filter(pl.element().ne(''))\
        .list.unique(maintain_order=True)


def tokenize_filtered(expr: pl.Expr) -> pl.Expr:
    return (
        tokenize_unique(expr)
        # Filter our years & BPM ranges
        .list.filter(~pl.element().str.contains("^([0-9]+|[0-9]+-[0-9]+)$"))
        # Filter out stuff consisting only of non-letters
        .list.filter(pl.element().str.contains("[[:alpha:]]"))
    )

Specific tokenization based on known single-word and multi-word keywords:

In [ ]:
def extract_tags_from_name(expr: pl.Expr) -> pl.Expr:
    """"Extract a list of tags from the given playlist name."""
    def create_regex_for_term(term: str) -> str:
        escaped_term = pl.escape_regex(term)
        # Ignore additional whitespaces (e.g. "a b" should also match "a  b")
        escaped_term = escaped_term.replace(' ', ' +')
        return f'\\b{escaped_term}\\b'

    alias_to_tags = load_keyword_aliases()
    all_keywords_alts = '|'.join([create_regex_for_term(term) for term in alias_to_tags])
    all_keywords_regex = f'(?i)({all_keywords_alts})'

    # Use regexes to extract the keywords, then match the
    # extracted strings against our dictionary to check
    # if the matched keyword should be aliased to something else
    return expr\
        .str.extract_all(all_keywords_regex)\
        .list.eval(pl.element()
                   .str.to_lowercase()
                   .replace_strict(alias_to_tags,
                                   default=pl.lit([], dtype=pl.List(pl.String)),
                                   return_dtype=pl.List(pl.String))
                   .explode())\
        .list.drop_nulls()\
        .list.unique()\
        .list.sort()

## Playlist statistics

### Extract known keywords from playlists

In [ ]:
playlists_tokenized = playlists.select(
    pl.col('playlist.id'),
    pl.col('playlist.name'),
    pl.col('playlist.name').pipe(tokenize_filtered).alias('terms'),
    pl.col('playlist.name').pipe(extract_tags_from_name).alias('tags'),
)

exploded_playlists_by_term = playlists_tokenized\
    .explode('terms')\
    .rename({'terms': 'term'})

terms = exploded_playlists_by_term\
    .group_by('term')\
    .agg(pl.col('term').count().alias('playlist_count'),
         pl.col('playlist.name').head(20))\
    .sort('playlist_count', descending=True)

exploded_playlists_by_tag = playlists_tokenized\
    .explode('tags')\
    .rename({'tags': 'tag'})

tags = exploded_playlists_by_tag\
    .group_by('tag')\
    .agg(pl.col('tag').count().alias('playlist_count'),
         pl.col('playlist.name').head(20))\
    .select(pl.col('tag').str.split(':').list.get(0).alias('category'),
            pl.col('tag').str.split(':').list.get(1, null_on_oob=True).alias('tag'),
            pl.col('tag').alias('full_tag'),
            'playlist_count',
            'playlist.name')\
    .sort('playlist_count', descending=True)

(tags
    # .filter(pl.col('playlist_count').ge(20))
    # .with_columns(pl.col('playlist.name').list.join("|"))
    # .sink_csv('out.csv', engine='streaming')
    .collect(engine='streaming'))

In [ ]:
playlists_tokenized.filter(pl.col('tags').list.drop_nulls().list.len().eq(1)).collect(engine='streaming')

playlist.id,playlist.name,terms,tags
str,str,list[str],list[str]
"""000QVDaEeEmfVJpfddp611""","""Songs with Swing/Blues Shuffle…","[""songs"", ""with"", … ""timing""]","[""genre:blues""]"
"""001yS3M8MpTH2qElXS8ODj""","""Pop Radio WCS""","[""pop"", ""radio"", ""wcs""]","[""genre:pop""]"
"""002xChxzGm2JXveOHYvzrb""","""Routine Songs""","[""routine"", ""songs""]","[""context:routine""]"
"""00BrxSphwuS0d6xb4iI0b5""","""Best WCS by BPM Old""","[""best"", ""wcs"", … ""old""]","[""genre:old""]"
"""00D9LlZLecTXiwkbe9rdPO""","""WCS Late night wcs""","[""wcs"", ""late"", ""night""]","[""genre:late night""]"
…,…,…,…
"""1oDS0BLwANmfjyL0f6IrH8""","""WCS - Fast (121 - 140 BPM)""","[""wcs"", ""fast"", ""bpm)""]","[""genre:fast""]"
"""1oG9Deh8tf9ArWe9Lt38E7""","""Late night""","[""late"", ""night""]","[""genre:late night""]"
"""1oGTe2lrUvSVgzmsYV3pNP""","""Blues""","[""blues""]","[""genre:blues""]"


### Scanning for additional patterns

In [ ]:
import itertools

patterns["misc"] = pattern_misc = [
    # These may warrant further investigation
    "beat",
    "danceability",
    "drops",
    "flow",
    "level",
    "tempo",
    "low",
    "high",
    " / ",

    # Sorting
    "decreasing",
    "increasing",
    "level",
    "ordered",
    "similar",
    "sorted",

    # Probably not WCS
    "tik", "tok",
]

patterns_new_exclude = [
    "down",  # "slow down", "steady down beat, slow",
    "first",  # "first half", "first hour",
    "school",  # "old school", "middle school",

    "white",
    "rabbit",
    "rebels",  # unrelated
    "fun",  # imprecise
    "star",  # imprecise
    "warm",  # imprecise

    # context 2
    "solo",
    "ballet",

    "new",
    "neu",
    "now",
    "hot",
    "current",
]

patterns_new = [
    "med",
    "feel",
    "feels",
    "groove",
    "mood",
    "connect",
    "intro",
    "cool",
    "soundtrack",
    "vibe",
    "sing",
    "pre",
    "freestyle",
    "weekend",
    "classical",
    "mid",
    "musical",
    "creativity",
    "kids",
    "super",  # "super fast",
    "very",
    "last",
    "first",
    "movement",
    "karaoke",
    "break",
    "early",
    "epic",
    "requests",
    "breaks",
    "long",
    "other",
    "min",
    "fire",
    "sa",
    "maybe",
    "when",
    "little",
    "bad",
    "too",
    "vs",
    "better",
    "half",
    "made",
    "going",
    "before",
    "along",
    "start",
    "crazy",
    "right",
    "water",
    "shower",
    "magic",
    "never",
    "ok",
    "really",
    "different",
    "dreams",
    "remember",
    "taste",
    "let's",
    "fantasy",
    "still",
    "welcome",
    "than",
    "boom",
    "keep",
    "same",
    "away",
    "should",
    "inspired",
    "pretty",
    "non",
    "mit",
    "getting",
    "heart",
    "gold",
    "real",
    "check",
    "close",
]

patterns_filtered = [
    "basic",
    "basics",
    # "connection", # needs further filtering
    "drill", "drills",
    "footwork",
    "lyrical",
    "phrase", "change", "phrase change", "phrasing",
    "rhythm", "rhythms",
    "structure", "bar", "count",
    "style", "styling",
    "timing",
    # #
    # "body",
    # "changes",
    # "critique",
    # "double",
    # "intro",
    # "intros",
    # "micro",
    # "mixed",
    # "moves",
    # "musique",
    # "performance",
    # "rounds",
    # "single",
    # "stretch",
    # "switch",

    # "together",
]

(terms
    .filter(pl.col('term').is_in(patterns_filtered))
    # .filter(~pl.col('term').is_in(list(itertools.chain.from_iterable(patterns.values()))))
    # .filter(~pl.col('term').is_in(patterns_new_exclude))
    # .filter(~pl.col('term').is_in(patterns_new))
    # .filter(pl.col('playlist_count').ge(20))
    # .with_columns(pl.col('playlist.name').list.join("|"))
    # .sink_csv('out.csv', engine='streaming')
    .collect(engine='streaming'))

term,playlist_count,playlist.name
str,u32,list[str]
"""rhythm""",35,"[""Clear ""Swung"" Rhythm"", ""3AW3-Swung Rhythm"", … ""WCS Swung Rhythm""]"
"""lyrical""",24,"[""Lyrical"", ""WCS Lyrical"", … ""Lyrical Fusion""]"
"""drills""",18,"[""Drills"", ""Adv drills UK"", … ""WCS Rythyms Drills""]"
"""footwork""",16,"[""Footwork"", ""Footwork"", … ""Footwork (115-126bpm)""]"
"""styling""",14,"[""Norway Arm styling "", ""Ladies Styling"", … ""Ladies styling hands 12.03""]"
…,…,…
"""structure""",6,"[""6x8 Blues Structure"", ""Camie blues structure not blues musicality"", … ""WCS Blues Structure""]"
"""bar""",5,"[""Find Another Bar"", ""12 bar blues"", … ""Musik 32 Bar AB (aka AA’)""]"
"""basic""",5,"[""Basic 🏡 "", ""L2 Basic Musicality"", … ""Basic Bitch by phoenixfaery""]"


## Song statistics

### Tag <=> Song correlations

In [38]:
relevant_tags = tags\
    .collect(engine='streaming')

track_tags = playlist_tracks\
    .join(exploded_playlists_by_tag
          # .join(relevant_tags.lazy(), how='semi', left_on='tag', right_on='full_tag')
          .filter(pl.col('tag').is_not_null()),
          how='inner', on='playlist.id')\
    .group_by('track.id', 'tag')\
    .agg(pl.col('tag').count().alias('playlist_count'))

temp_file = 'temp_track_tags.parquet'
track_tags.sink_parquet(temp_file)
track_tags = pl.scan_parquet(temp_file)

track_tags.limit(10).collect(engine='streaming')

track.id,tag,playlist_count
str,str,u32
"""1QpcKwlEjFH25d6KIJmNdm""","""topic:timing""",1
"""2FsnLNwjEsrVN8bCVsOh0N""","""timing:shuffle""",1
"""3z2hQfmtRAtNkBa61sBt91""","""genre:blues""",1
"""5uYIDLDDuF6pEfyaJNt995""","""topic:timing""",1
"""6Vc5wAMmXdKIAM7WUoEb7N""","""topic:timing""",1
"""7mtL1xNVz7uZvlMLav5a7f""","""topic:timing""",1
"""6Xxoo5GaI3RRj41EPODOeW""","""context:party""",46
"""6bQ7N7EE3MxvDivQlPiEkx""","""context:party""",1
"""5m3F7x3wk7S3Zm5mzc0gXP""","""genre:country""",9


In [39]:
matching_tracks = tracks\
    .filter(pl.col('track.name').eq('Valleys'))\
    .select('track.id', 'track.artists', 'track.name')

matching_playlist_tracks = playlist_tracks\
    .join(matching_tracks, how='semi', on='track.id')\

matching_playlists = playlists\
    .join(matching_playlist_tracks, how='semi', on='playlist.id')\
    .filter(pl.col('playlist.name').str.contains('(?i)late night'))

matching_playlists.collect()

playlists_tokenized.join(matching_playlists, how='semi', on='playlist.id').collect()

playlist.id,playlist.name,terms,tags
str,str,list[str],list[str]
"""00D9LlZLecTXiwkbe9rdPO""","""WCS Late night wcs""","[""wcs"", ""late"", ""night""]","[""genre:late night""]"
"""0UF5i5Zi2fWo9M0JHhXRPG""","""Late night WCS vibes 😪😴""","[""late"", ""night"", … ""vibes""]","[""genre:late night"", ""mood:vibes""]"
"""0XjVGgbFdOQ5oHryDvVce8""","""SES 2022 - Late Night Sat/Sun""","[""ses"", ""late"", … ""sat/sun""]","[""genre:late night""]"
"""0fdyEyAEG9Nt6oOJlahQc0""","""Late Night WCS""","[""late"", ""night"", ""wcs""]","[""genre:late night""]"
"""12vJkTvJE4nvl63JdAKjDT""","""wny late night sept 18""","[""wny"", ""late"", … ""sept""]","[""genre:late night""]"
"""13sid7Y2TGoYBsbuFkhiYM""","""SHAYS LATE NIGHT DDA""","[""shays"", ""late"", … ""dda""]","[""genre:late night""]"
"""1CVKelParw0txBGjAb5WzS""","""West coast late night""","[""west"", ""coast"", … ""night""]","[""genre:late night""]"
"""1HM39YFYqFk3agOfbhO5Ht""","""late late night wcs""","[""late"", ""night"", ""wcs""]","[""genre:late night""]"
"""1KFk2VKW1EyOhr0fh7oxSN""","""WCS Late Night Songs""","[""wcs"", ""late"", … ""songs""]","[""genre:late night""]"


In [40]:
temp_file = 'temp_track_tags_by_track_id.parquet'
track_tags_by_track_id = track_tags.sort('track.id')
track_tags_by_track_id.sink_parquet(temp_file)
track_tags_by_track_id = pl.scan_parquet(temp_file)

Using `group_by` to aggregate a column into a list is currently not supported by Polars' `streaming` engine.
To avoid crashing with an OOM, we sequentially process batches of `track.id`s instead:

In [43]:
import math


def process_track_tags_batch(tracks_batch: pl.LazyFrame) -> pl.LazyFrame:
    return track_tags_by_track_id\
        .join(tracks_batch, how='semi', on='track.id')\
        .group_by('track.id')\
        .agg(pl.col('tag').sort_by('playlist_count', descending=True).head(20),
             pl.col('playlist_count').sort(descending=True).head(20).alias('playlist_counts'),
             pl.col('playlist_count').sort(descending=True).head(20).sum())\
        .join(tracks_batch.select('track.id', 'track.name', 'track.artists'), how='inner', on='track.id')


def process_track_tags_in_batches():
    row_count = tracks.select(pl.len()).collect().item()
    batch_size = 10000  # Higher batch sizes are faster but have a righer OOM risk
    batch_count = int(math.ceil(row_count / batch_size))

    print(f"Processing {row_count:,} tracks in {batch_count:,} batches of {batch_size:,} items...")

    for batch_index in range(0, batch_count):
        batch_start = batch_index * batch_size
        print(f"Processing batch {batch_index:,}/{batch_count:,}")
        batch_result = process_track_tags_batch(tracks.slice(batch_start, batch_size))\
            .sort('track.id')
        batch_result.sink_parquet(f'temp_tag_batch_{batch_index}.parquet')

    print("Merging batches...")

    merged: pl.LazyFrame | None = None
    for batch_index in range(0, batch_count):
        batch_data = pl.scan_parquet(f'temp_tag_batch_{batch_index}.parquet')
        merged = (batch_data if merged is None else
                  merged.merge_sorted(batch_data, 'track.id'))

    merged.sink_parquet('temp_tags_by_track.parquet')

    print("Done.")


process_track_tags_in_batches()

Processing 203,709 tracks in 21 batches of 10,000 items...
Processing batch 0/21
Processing batch 1/21
Processing batch 2/21
Processing batch 3/21
Processing batch 4/21
Processing batch 5/21
Processing batch 6/21
Processing batch 7/21
Processing batch 8/21
Processing batch 9/21
Processing batch 10/21
Processing batch 11/21
Processing batch 12/21
Processing batch 13/21
Processing batch 14/21
Processing batch 15/21
Processing batch 16/21
Processing batch 17/21
Processing batch 18/21
Processing batch 19/21
Processing batch 20/21
Merging batches...
Done.


In [42]:
tags_by_track = pl.scan_parquet('temp_tags_by_track.parquet')

song_tag_confidence = tags_by_track\
    .explode('tag', 'playlist_counts')\
    .filter(pl.col('tag').str.contains(':'))\
    .filter(pl.col('tag').eq('genre:acoustic'))\
    .rename({'playlist_count': 'track.sum_of_playlist_count_over_all_tags',
             'playlist_counts': 'matching_playlist_count'})\
    .join(tracks.select('track.id', 'playlist_count'), how='inner', on='track.id')\
    .join(tags.select('full_tag', pl.col('playlist_count').alias('tag.playlist_count')),
          how='inner', left_on='tag', right_on='full_tag')\
    .rename({'playlist_count': 'track.playlist_count'})\
    .sort('matching_playlist_count', descending=True)\
    .select('track.id',
            'tag',
            'matching_playlist_count',
            (pl.col('matching_playlist_count') / pl.col('tag.playlist_count')).alias('tag.playlist_percent'),
            'tag.playlist_count',
            # TODO: The best metric would probably be to compare matching_playlist_count to
            #       the number of playlists with this track that have at least one genre tag
            (pl.col('matching_playlist_count') / pl.col('track.playlist_count')).alias('track.playlist_percent'),
            'track.playlist_count',
            'track.name',
            'track.artists')

song_tag_confidence.sink_parquet('acoustic_tags.parquet')

song_tag_confidence\
    .rename({'matching_playlist_count': '# of playlists with track + tag',
             'tag.playlist_count': '# of playlists with tag',
             'tag.playlist_percent': 'track in % of playlists with tag',
             'track.playlist_count': '# of playlists with track',
             'track.playlist_percent': 'tag in % of playlists with track'})\
    .collect(engine='streaming')

track.id,tag,# of playlists with track + tag,track in % of playlists with tag,# of playlists with tag,tag in % of playlists with track,# of playlists with track,track.name,track.artists
str,str,u32,f64,u32,f64,u32,str,list[str]
"""2HF9OydUs28eTuiVUjQYdT""","""genre:acoustic""",14,0.189189,74,0.011966,1170,"""Things Are Changin' - Solo Aco…","[""Gary Clark Jr.""]"
"""1kJygfS4eoVziBBI93MSYp""","""genre:acoustic""",13,0.175676,74,0.004569,2845,"""Acoustic""","[""Billy Raffoul""]"
"""3Sq9O2GzAmWQSFyI5oLX8F""","""genre:acoustic""",11,0.148649,74,0.006474,1699,"""The Way You Make Me Feel""","[""Kurt Hunter""]"
"""5DHvDSGFyNI02zDeQSJ07V""","""genre:acoustic""",11,0.148649,74,0.014066,782,"""Work The Middle - Acoustic""","[""Alex Aiono""]"
"""16YdPF3ZGFIfouwajAhv6C""","""genre:acoustic""",11,0.148649,74,0.015257,721,"""Boyfriend - Acoustic Version""","[""Justin Bieber""]"
…,…,…,…,…,…,…,…,…
"""1PiKcVhlmHPvL7IPjx9dJk""","""genre:acoustic""",1,0.013514,74,0.004167,240,"""I Took A Pill In Ibiza""","[""Mike Posner""]"
"""1QttDfIdwAgyNFB45nUDDp""","""genre:acoustic""",1,0.013514,74,0.03125,32,"""Modern Flame""","[""Yuna"", ""Emmit Fenn""]"
"""1RANp7wwRD1ecNmn6I3MI4""","""genre:acoustic""",1,0.013514,74,0.5,2,"""Tell Me A Story""","[""Phillip Phillips""]"
